# 06 — One analysis-ready cube

Merges everything this project produces into a single NetCDF on one grid:

| source | notebook | variables |
|---|---|---|
| GloFAS-ERA5 v4.0 | `01` | `discharge`, `runoff`, `soil_wetness` |
| MODIS MCDWD_L3 | `05` | `flood_fraction`, `flood_binary`, `valid_fraction` |

Everything is stored in **physical units**. Standardization is a modelling choice, not a property of
the data, so the per-channel mean/std travel as attributes and can be reapplied at training time
instead of being baked in.

The merge **asserts** that the grids and time axes agree rather than reindexing onto each other —
if a future rerun changes the ROI or window in one place, this notebook fails loudly instead of
silently interpolating.

In [1]:
# ============================================================
# CELL 1 - Load the pieces and check they really are on one grid
# ============================================================
from pathlib import Path
import glob, json

import numpy as np
import xarray as xr

NC_ENGINE = "h5netcdf"          # netCDF4 in this env is built against numpy 1.x
OUT_DIR   = Path("update/analysis_ready"); OUT_DIR.mkdir(parents=True, exist_ok=True)

raw   = xr.open_dataset(glob.glob("update/glofas_ready/*_raw.nc")[0],  engine=NC_ENGINE)
lab   = xr.open_dataset(glob.glob("update/flood_labels/*.nc")[0],      engine=NC_ENGINE)
stats = json.load(open(glob.glob("update/glofas_ready/*_stats.json")[0]))

# Hard checks - never silently realign.
assert np.array_equal(raw.latitude.values,  lab.latitude.values),  "latitude grids differ"
assert np.array_equal(raw.longitude.values, lab.longitude.values), "longitude grids differ"
assert np.array_equal(raw.time.values,      lab.time.values),      "time axes differ"
print("grid check passed")
print(f"  time {str(raw.time.values[0])[:10]} .. {str(raw.time.values[-1])[:10]}  ({raw.time.size} days)")
print(f"  grid {raw.latitude.size} x {raw.longitude.size} @ "
      f"{abs(float(raw.latitude[1]-raw.latitude[0])):.2f} deg")

grid check passed
  time 2022-05-01 .. 2022-08-31  (123 days)
  grid 130 x 140 @ 0.05 deg


/opt/anaconda3/envs/sia/lib/python3.11/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'gmt' loading failed:
Error loading GMT shared library at 'libgmt.dylib'.
dlopen(libgmt.dylib, 0x0006): tried: 'libgmt.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibgmt.dylib' (no such file), '/opt/anaconda3/envs/sia/lib/python3.11/lib-dynload/../../libgmt.dylib' (no such file), '/opt/anaconda3/envs/sia/bin/../lib/libgmt.dylib' (no such file), '/usr/lib/libgmt.dylib' (no such file, not in dyld cache), 'libgmt.dylib' (no such file), '/usr/local/lib/libgmt.dylib' (no such file), '/usr/lib/libgmt.dylib' (no such file, not in dyld cache)
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)


In [2]:
# ============================================================
# CELL 2 - Merge
# ============================================================
UNITS = {"discharge":    ("m3 s-1", "time-mean river discharge (GloFAS v4.0)"),
         "runoff":       ("mm day-1", "runoff water equivalent (GloFAS v4.0)"),
         "soil_wetness": ("1", "soil wetness index (GloFAS v4.0)")}

ds = xr.Dataset(coords={"time": raw.time, "latitude": raw.latitude, "longitude": raw.longitude})

# GloFAS: split the channel dimension into named variables - far easier to use than an index.
for ch in raw.channel.values:
    u, name = UNITS[str(ch)]
    ds[str(ch)] = (("time", "latitude", "longitude"),
                   raw.glofas.sel(channel=ch).values.astype("float32"),
                   {"units": u, "long_name": name, "source": "GloFAS-ERA5 v4.0 (EWDS)",
                    "norm_mean": stats[str(ch)]["mean"], "norm_std": stats[str(ch)]["std"]})

# MODIS labels, carried over with their own attributes intact.
for v in ("flood_fraction", "flood_binary", "valid_fraction"):
    ds[v] = (("time", "latitude", "longitude"), lab[v].values, dict(lab[v].attrs))
    ds[v].attrs["source"] = "MODIS MCDWD_L3 v061 (NASA LAADS DAAC)"

ds.attrs = {
    "title": "Analysis-ready flood cube - Bangladesh + NE India, May-Aug 2022",
    "region": "bd_ne_india", "area_NWSE": "27.5, 87.0, 21.0, 94.0",
    "resolution_deg": 0.05, "time_step": "daily",
    "inputs": "GloFAS-ERA5 v4.0 (discharge, runoff, soil_wetness)",
    "labels": "MODIS MCDWD_L3 v061 (flood_fraction, flood_binary, valid_fraction)",
    "units_note": "all variables in physical units; norm_mean/norm_std are attributes on each "
                  "GloFAS variable so standardization can be applied at training time",
    "nodata_note": "flood_binary = -1 and flood_fraction = NaN where MODIS had too few valid "
                   "pixels; valid_fraction records how much of each cell was actually observed",
    "built_by": "06_analysis_ready_cube.ipynb",
}
print(ds)

<xarray.Dataset> Size: 47MB
Dimensions:         (time: 123, latitude: 130, longitude: 140)
Coordinates:
    surface         float64 8B 0.0
    rootZone        float64 8B 0.0
  * time            (time) datetime64[ns] 984B 2022-05-01 ... 2022-08-31
  * latitude        (latitude) float64 1kB 27.48 27.43 27.38 ... 21.07 21.02
  * longitude       (longitude) float64 1kB 87.03 87.08 87.12 ... 93.92 93.97
Data variables:
    discharge       (time, latitude, longitude) float32 9MB 7.094 ... 33.82
    runoff          (time, latitude, longitude) float32 9MB 10.14 ... 8.114
    soil_wetness    (time, latitude, longitude) float32 9MB 0.8302 ... 0.9423
    flood_fraction  (time, latitude, longitude) float32 9MB 0.0 0.0 ... 0.0 0.0
    flood_binary    (time, latitude, longitude) int8 2MB 0 0 0 0 0 ... 0 0 0 0 0
    valid_fraction  (time, latitude, longitude) float32 9MB 1.0 1.0 ... 1.0 1.0
Attributes:
    title:           Analysis-ready flood cube - Bangladesh + NE India, May-A...
    region:       

In [3]:
# ============================================================
# CELL 3 - Save + manifest
# ============================================================
tag = f"bd_ne_india_{str(ds.time.values[0])[:10]}_to_{str(ds.time.values[-1])[:10]}"
nc  = OUT_DIR / f"flood_cube_{tag}.nc"

enc = {v: {"zlib": True, "complevel": 4} for v in ds.data_vars}
ds.to_netcdf(nc, engine=NC_ENGINE, encoding=enc)

man = {
    "tag": tag,
    "shape_time_lat_lon": [ds.sizes["time"], ds.sizes["latitude"], ds.sizes["longitude"]],
    "resolution_deg": 0.05,
    "area_NWSE": [27.5, 87.0, 21.0, 94.0],
    "time_start": str(ds.time.values[0])[:10], "time_end": str(ds.time.values[-1])[:10],
    "inputs": [v for v in ("discharge", "runoff", "soil_wetness") if v in ds],
    "labels": [v for v in ("flood_fraction", "flood_binary", "valid_fraction") if v in ds],
    "norm_stats": {k: {"mean": stats[k]["mean"], "std": stats[k]["std"]} for k in stats},
    "usable_label_fraction": float(np.isfinite(ds.flood_fraction.values).mean()),
    "file": nc.name,
}
(OUT_DIR / f"flood_cube_{tag}_manifest.json").write_text(json.dumps(man, indent=2))
print(f"wrote {nc}  ({nc.stat().st_size/1e6:.1f} MB)")

wrote update/analysis_ready/flood_cube_bd_ne_india_2022-05-01_to_2022-08-31.nc  (19.0 MB)


In [4]:
# ============================================================
# CELL 4 - Reopen and verify what a downstream user would get
# ============================================================
chk = xr.open_dataset(nc, engine=NC_ENGINE)
print(f"variables : {list(chk.data_vars)}")
print(f"dims      : {dict(chk.sizes)}\n")
for v in chk.data_vars:
    a = chk[v].values.astype("float64")
    a = np.where(a == -1, np.nan, a) if v == "flood_binary" else a
    print(f"  {v:16s} {str(chk[v].dtype):8s} "
          f"min {np.nanmin(a):10.3f}  max {np.nanmax(a):10.3f}  "
          f"NaN {np.isnan(a).mean()*100:5.1f}%  units={chk[v].attrs.get('units','-')}")

# The model-ready array, assembled the way training would do it.
X = np.stack([chk[v].values for v in ("discharge", "runoff", "soil_wetness")], axis=1)
y = chk.flood_fraction.values
print(f"\nX (time, channel, lat, lon) = {X.shape}")
print(f"y (time, lat, lon)          = {y.shape}")

variables : ['discharge', 'runoff', 'soil_wetness', 'flood_fraction', 'flood_binary', 'valid_fraction']
dims      : {'time': 123, 'latitude': 130, 'longitude': 140}

  discharge        float32  min      0.000  max 104540.281  NaN   0.0%  units=m3 s-1
  runoff           float32  min      0.000  max    222.754  NaN   0.0%  units=mm day-1
  soil_wetness     float32  min      0.205  max      0.987  NaN   0.0%  units=1
  flood_fraction   float32  min      0.000  max      1.000  NaN  58.0%  units=1
  flood_binary     int8     min      0.000  max      1.000  NaN  58.0%  units=-
  valid_fraction   float32  min      0.000  max      1.000  NaN   0.0%  units=1

X (time, channel, lat, lon) = (123, 3, 130, 140)
y (time, lat, lon)          = (123, 130, 140)
